In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [ ]:
!pip install -q -U transformers accelerate gradio

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("Selected Model:", MODEL_NAME)

Selected Model: HuggingFaceTB/SmolLM2-1.7B-Instruct


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("LLM loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LLM loaded successfully!


In [ ]:
print("Model device:", model.device)

Model device: cuda:0


In [ ]:


conversation = [
    {
        "role": "system",
        "content": "You are a helpful, friendly and knowledgeable AI assistant."
    }
]


def chatbot(user_message):

    # Add user message to conversation
    conversation.append({
        "role": "user",
        "content": user_message
    })

    # Convert conversation into model inputs
    inputs = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Move inputs to the same device as model
    inputs = inputs.to(model.device)

    # Generate response
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )

    # Find where the input ends
    input_length = inputs["input_ids"].shape[-1]

    # Extract only newly generated tokens
    new_tokens = outputs[0][input_length:]

    # Convert tokens into text
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    # Save assistant response
    conversation.append({
        "role": "assistant",
        "content": response
    })

    return response

In [ ]:
conversation = [
    {
        "role": "system",
        "content": "You are a helpful, friendly and knowledgeable AI assistant."
    }
]

print("Conversation reset successfully!")

Conversation reset successfully!


In [ ]:
response = chatbot("Hello! Introduce yourself.")

print("AI:", response)

AI: Hello! My name is LanguageMaster, and I'm an advanced language learning AI designed to assist users in improving their understanding and use of various languages. I'm here to help you with grammar, vocabulary, pronunciation, and much more. Feel free to ask me any questions or request assistance on your language learning journey.


In [ ]:
response = chatbot("What is artificial intelligence?")

print("AI:", response)

AI: Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks requiring human-like intelligence, such as learning from experience, understanding complex concepts, recognizing patterns, solving problems, and making decisions. These systems use algorithms and data processing capabilities to mimic human thought processes, allowing them to make predictions, learn from new information, and adapt to different situations.

Think of AI like a very smart virtual assistant who can process vast amounts of information quickly, recognize speech or text, understand context, and provide useful insights or recommendations. AI has numerous applications across industries, including healthcare, finance, transportation, education, and entertainment. It's transforming many aspects of our lives by enabling machines to work alongside humans, enhancing efficiency, productivity, and innovation.


In [ ]:
response = chatbot("Explain it in simple words.")

print("AI:", response)

AI: Imagine having a super smart helper that can learn things for you, think like you do, and solve problems just like you would. That's basically what Artificial Intelligence (AI) does. It's made possible through computers and technology that allow them to learn, reason, and act like humans, helping us in lots of ways.


In [ ]:
import gradio as gr

print("Gradio Version:", gr.__version__)

Gradio Version: 6.26.0


In [ ]:
def generate_response(message, history):

    # Start conversation with system instruction
    messages = [
        {
            "role": "system",
            "content": "You are a helpful, friendly and knowledgeable AI assistant."
        }
    ]

    # Add previous conversation
    if history:

        for item in history:

            # Handle dictionary format
            if isinstance(item, dict):

                role = item.get("role")
                content = item.get("content")

                if role in ["user", "assistant"] and content:
                    messages.append({
                        "role": role,
                        "content": str(content)
                    })

            # Handle old tuple/list format
            elif isinstance(item, (list, tuple)):

                if len(item) >= 2:

                    user_message = item[0]
                    assistant_message = item[1]

                    if user_message:
                        messages.append({
                            "role": "user",
                            "content": str(user_message)
                        })

                    if assistant_message:
                        messages.append({
                            "role": "assistant",
                            "content": str(assistant_message)
                        })

    # Add current user message
    messages.append({
        "role": "user",
        "content": str(message)
    })

    # Convert conversation to model input
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Move inputs to model device
    inputs = inputs.to(model.device)

    # Generate response
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )

    # Get input length
    input_length = inputs["input_ids"].shape[-1]

    # Get only newly generated tokens
    new_tokens = outputs[0][input_length:]

    # Convert tokens to text
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return response

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("Libraries loaded!")

Libraries loaded!


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
print(type(tokenizer))

<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>


In [ ]:
print("Tokenizer:", "Loaded" if 'tokenizer' in globals() else "Missing")
print("Model:", "Loaded" if 'model' in globals() else "Missing")

Tokenizer: Loaded
Model: Missing


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("LLM loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LLM loaded successfully!


In [ ]:
print("Model device:", model.device)

Model device: cuda:0


In [ ]:
test_response = generate_response(
    "What is machine learning?",
    []
)

print("AI:", test_response)

AI: Machine learning is a subset of artificial intelligence that allows systems to automatically learn from data and improve their performance without being explicitly programmed. It involves the use of algorithms or statistical models to enable computers to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.

The core idea behind machine learning is that these algorithms can identify patterns in large datasets, make predictions based on those patterns, and even generate new insights or hypotheses. The more data a system has access to, the better it becomes at making accurate predictions or decisions. This process is often referred to as "training" a model because the algorithm learns from the data during this training phase. 

There are several types of machine learning, including supervised learning (where the model learns from labeled data), unsupervised learning (where the model identifie

In [ ]:
import gradio as gr

print("Gradio Version:", gr.__version__)

Gradio Version: 6.26.0


In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=generate_response,
    title="🤖 My AI Assistant",
    description="💬 LLM-Powered Chatbot using SmolLM2-1.7B-Instruct",
    textbox=gr.Textbox(
        placeholder="Ask me anything...",
        lines=1
    ),
    submit_btn="Send 🚀"
)

print("Chatbot interface created successfully!")

Chatbot interface created successfully!


/tmp/ipykernel_1733/172377857.py:3: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  demo = gr.ChatInterface(


In [ ]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://360d5d354b8106c1b8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
